In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve
import struct

class TetraMeshProcessor:
    """
    Класс для работы с тетраэдральными сетками:
    расчеты методом конечных элементов и сжатие данных.
    """
    def __init__(self, vertices, tets):
        # Используем float64 для хранения вершин по умолчанию
        self.vertices = np.array(vertices, dtype=np.float64) # Изменено на float64
        self.tets = np.array(tets, dtype=np.int32)
        self.num_vertices = len(vertices)
        self.num_tets = len(tets)

    # --- Секция TetraFEM ---

    def compute_element_stiffness(self, tet_idx):
        """
        Вычисляет локальную матрицу жесткости 4x4 для одного тетраэдра.
        Используется для уравнения Пуассона: -div(grad(u)) = f
        """
        nodes = self.vertices[self.tets[tet_idx]]

        # Матрица координат для вычисления объема и градиентов
        # Строим матрицу [1, x, y, z] для каждой вершины
        M = np.ones((4, 4))
        M[:, 1:] = nodes

        volume = np.abs(np.linalg.det(M)) / 6.0
        if volume < 1e-15:
            return np.zeros((4, 4))

        # Вычисляем обратную матрицу для поиска коэффициентов базисных функций
        inv_M = np.linalg.inv(M)
        # Градиенты базисных функций (b, c, d коэффициенты)
        grads = inv_M[1:, :] # (3, 4) - градиенты по x, y, z для 4-х узлов

        # Локальная матрица жесткости K_ij = Integral(grad(phi_i) * grad(phi_j))
        # Для линейных функций градиент постоянен: K_ij = volume * (grad_i dot grad_j)
        element_stiffness = volume * (grads.T @ grads)

        return element_stiffness

    def assemble_global_stiffness(self):
        """
        Сборка глобальной разреженной матрицы жесткости.
        """
        I, J, Data = [], [], []

        for idx in range(self.num_tets):
            ke = self.compute_element_stiffness(idx)
            nodes = self.tets[idx]

            for i in range(4):
                for j in range(4):
                    I.append(nodes[i])
                    J.append(nodes[j])
                    Data.append(ke[i, j])

        return sp.csc_matrix((Data, (I, J)), shape=(self.num_vertices, self.num_vertices))

    # --- Секция Сжатия (Compression) ---

    def compress_mesh(self, bits=12):
        """
        Сжатие сетки с использованием квантования и дельта-кодирования.
        Использует float32 для промежуточных вычислений
        """
        # 1. Квантование (Quantization)
        # Преобразуем float64 в float32
        vertices_f32 = self.vertices.astype(np.float32)

        v_min = vertices_f32.min(axis=0)
        v_max = vertices_f32.max(axis=0)
        scale = (2**bits - 1) / (v_max - v_min)

        quantized_v = np.round((vertices_f32 - v_min) * scale).astype(np.int32)

        # 2. Дельта-кодирование (Delta Encoding) для координат
        # Вместо [v0, v1, v2...] храним [v0, v1-v0, v2-v1...]
        deltas = np.zeros_like(quantized_v)
        deltas[0] = quantized_v[0]
        deltas[1:] = quantized_v[1:] - quantized_v[:-1]

        # 3. Упаковка в бинарный поток (пример)
        # Здесь можно использовать энтропийное кодирование (Хаффман или Arithmetic)
        compressed_data = {
            'min': v_min, # Эти значения будут float32
            'max': v_max, # Эти значения будут float32
            'deltas': deltas,
            'topology': self.tets # Топологию можно сжать через Edge-breaker
        }
        return compressed_data

    @staticmethod
    def decompress_mesh(compressed_data, bits=12):
        """
        Восстановление сетки из сжатого состояния.
        """
        deltas = compressed_data['deltas']
        v_min = compressed_data['min'] # float32
        v_max = compressed_data['max'] # float32

        # Обратное дельта-кодирование (префиксная сумма)
        quantized_v = np.cumsum(deltas, axis=0)

        # Обратное квантование
        scale = (v_max - v_min) / (2**bits - 1)
        vertices = quantized_v * scale + v_min # Результат будет float32

        # Возвращаем в float64 для сравнения с оригинальными данными
        return vertices.astype(np.float64), compressed_data['topology'] # Изменено на float64

In [ ]:

# --- Пример работы ---
# Создаем большую сетку: 4x4x4 куба, 5x5x5 вершин

grid_resolution = 4 # Например, 4x4x4 куба, что дает 5x5x5 вершин

new_verts = []
for z in range(grid_resolution + 1):
    for y in range(grid_resolution + 1):
        for x in range(grid_resolution + 1):
            new_verts.append([float(x), float(y), float(z)])

new_indices = []
# Вспомогательная функция для получения индекса вершины по координатам (x,y,z)
def get_idx(x, y, z, res):
    return z * (res + 1)**2 + y * (res + 1) + x

# Декомпозиция каждого единичного куба на 6 тетраэдров
# Используется стандартная "тип-4" декомпозиция, где все 6 тетраэдров разделяют общую диагональ куба (от v0 до v7)
for k in range(grid_resolution):
    for j in range(grid_resolution):
        for i in range(grid_resolution):
            # Вершины текущего куба
            v0 = get_idx(i, j, k, grid_resolution)
            v1 = get_idx(i+1, j, k, grid_resolution)
            v2 = get_idx(i, j+1, k, grid_resolution)
            v3 = get_idx(i+1, j+1, k, grid_resolution)
            v4 = get_idx(i, j, k+1, grid_resolution)
            v5 = get_idx(i+1, j, k+1, grid_resolution)
            v6 = get_idx(i, j+1, k+1, grid_resolution)
            v7 = get_idx(i+1, j+1, k+1, grid_resolution)

            # 6 тетраэдров, образующих куб
            new_indices.append([v0, v1, v3, v7])
            new_indices.append([v0, v2, v3, v7])
            new_indices.append([v0, v4, v5, v7])
            new_indices.append([v0, v5, v1, v7])
            new_indices.append([v0, v6, v4, v7])
            new_indices.append([v0, v2, v6, v7])

processor = TetraMeshProcessor(new_verts, new_indices)

# 1. FEM Анализ
print("--- TetraFEM ---")
K = processor.assemble_global_stiffness()
print(f"Размер глобальной матрицы жесткости: {K.shape}")
print("Ненулевые элементы (первые 5):\n", K.toarray()[:3, :3])

# 2. Сжатие
print("\n--- Сжатие сетки ---")
bits_precision = 16
compressed = processor.compress_mesh(bits=bits_precision)

original_size = processor.vertices.nbytes
# Грубая оценка размера дельт (в реальности меньше после энтропийного кодирования)
compressed_size = compressed['deltas'].nbytes

print(f"Оригинальный размер вершин: {original_size} байт")
print(f"Размер после квантования (без энтропийного кодирования): {compressed_size} байт")

# 3. Проверка точности
v_rec, _ = processor.decompress_mesh(compressed, bits=bits_precision)
error = np.linalg.norm(processor.vertices - v_rec)
print(f"L2 ошибка восстановления при {bits_precision} битах: {error:.2e}")

--- TetraFEM ---
Размер глобальной матрицы жесткости: (125, 125)
Ненулевые элементы (первые 5):
 [[ 1.         -0.33333333  0.        ]
 [-0.33333333  1.66666667 -0.33333333]
 [ 0.         -0.33333333  1.66666667]]

--- Сжатие сетки ---
Оригинальный размер вершин: 3000 байт
Размер после квантования (без энтропийного кодирования): 1500 байт
L2 ошибка восстановления при 16 битах: 3.24e-04
